# Self Joins

## Overview
A self join joins a table to itself. It is used whenever a table contains a relationship between rows within the same table — typically hierarchies (manager/employee, supervisor/provider), peer relationships, or comparisons between rows.

**Syntax:** Give the same table two different aliases to treat it as two logical tables:
```sql
SELECT  a.col  AS col_a,
        b.col  AS col_b
FROM    mytable AS a
JOIN    mytable AS b ON a.some_key = b.other_key;
```

**Common use cases:**

| Pattern | Example |
|---|---|
| Hierarchy traversal | Provider → supervisor → department head |
| Peer comparison | Find patients seen by the same provider |
| Row-to-row comparison | Encounters on the same day |
| Finding duplicates | Rows with the same values on multiple columns |
| Sequential relationships | Previous/next appointment |

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    dob TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT,
    province TEXT, supervisor_id INTEGER
);
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_provider_id INTEGER
);
CREATE TABLE encounters (
    enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER,
    dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER DEFAULT 0
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE referrals (
    referral_id INTEGER PRIMARY KEY, from_provider INTEGER, to_provider INTEGER,
    patient_id INTEGER, referral_date TEXT, reason TEXT
);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','NB',1),(2,'Liam','Chen','1972-11-04','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','ON',1),(4,'James','MacLeod','1955-01-30','NB',0),
  (5,'Sofia','Petrov','2001-09-15','BC',1),(6,'Noah','Williams','1968-06-08','AB',1),
  (7,'Mei','Zhang','1995-04-17','ON',1),(8,'Ethan','Tremblay','1980-12-01','QC',0),
  (9,'Priya','Nair','1977-08-25','BC',1),(10,'Marcus','Okafor','1993-05-19','ON',1),
  (11,'Dana','Leblanc','2000-02-14','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),
  (11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Al-Rashid','Family Medicine','ON',10),
  (13,'Dr. Ethan Tremblay','Orthopaedics','QC',10),
  (14,'Dr. Priya Nair','Emergency Medicine','BC',11),
  (15,'Dr. Marcus Okafor','Radiology','ON',11),
  (16,'Dr. Unassigned','Neurology','NB',12);
INSERT INTO departments VALUES
  (1,'Cardiology','Building A',10),(2,'Oncology','Building B',11),
  (3,'Family Medicine','Building C',12),(4,'Orthopaedics','Building A',13),
  (5,'Emergency','Building D',14),(6,'Radiology','Building B',15),(7,'Neurology','Building C',16);
INSERT INTO diagnoses VALUES
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('Z00.0','General adult examination','Preventive'),
  ('M17.1','Primary osteoarthritis, knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Cardiovascular'),
  ('I10','Essential hypertension','Cardiovascular'),
  ('R55','Syncope and collapse','Neurological'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Musculoskeletal'),
  ('M54.5','Low back pain','Musculoskeletal');
INSERT INTO encounters VALUES
  (1,1,10,1,'2023-01-15','J18.9',1850.00,1),(2,2,10,1,'2023-02-20','I25.1',3200.00,1),
  (3,3,12,3,'2023-03-05','Z00.0',120.00,0),(4,4,13,4,'2023-03-18','M17.1',5500.00,1),
  (5,5,14,5,'2023-04-02','J06.9',95.00,0),(6,6,10,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,1,'2023-06-22','I10',2100.00,1),(8,8,12,3,'2023-07-14',NULL,80.00,0),
  (9,1,14,5,'2023-08-30','R55',410.00,0),(10,3,12,3,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,1,'2023-10-01','I48.0',1750.00,1),(12,10,13,4,'2023-11-03','S52.5',2200.00,1),
  (13,2,12,3,'2023-11-18',NULL,90.00,0),(14,5,13,4,'2023-12-05','M54.5',450.00,0);
INSERT INTO referrals VALUES
  (1,12,10,3,'2023-03-10','Chest pain follow-up'),
  (2,10,11,2,'2023-03-01','Suspected malignancy'),
  (3,14,10,9,'2023-09-05','AF workup'),
  (4,12,13,5,'2023-12-01','Back pain unresponsive to treatment'),
  (5,10,15,6,'2023-06-15','Imaging for chest pain');
""")
conn.commit()

print("providers table with supervisor_id hierarchy:")
print(pd.read_sql(
    "SELECT provider_id, full_name, specialty, supervisor_id FROM providers ORDER BY provider_id",
    conn).to_string(index=False))
print()
print("supervisor_id=NULL means top of hierarchy (no supervisor)")


providers table with supervisor_id hierarchy:
 provider_id            full_name          specialty  supervisor_id
          10    Dr. Sarah MacNeil         Cardiology            NaN
          11       Dr. James Wong           Oncology           10.0
          12 Dr. Fatima Al-Rashid    Family Medicine           10.0
          13   Dr. Ethan Tremblay       Orthopaedics           10.0
          14       Dr. Priya Nair Emergency Medicine           11.0
          15    Dr. Marcus Okafor          Radiology           11.0
          16       Dr. Unassigned          Neurology           12.0

supervisor_id=NULL means top of hierarchy (no supervisor)


---
## Self join for hierarchy: provider → supervisor

In [2]:
# Join providers to itself to show each provider alongside their supervisor
print("=== Provider hierarchy (provider → supervisor) ===")
q = """
SELECT  p.provider_id,
        p.full_name       AS provider,
        p.specialty,
        s.full_name       AS supervisor,
        s.specialty       AS supervisor_specialty
FROM    providers AS p
LEFT JOIN providers AS s ON p.supervisor_id = s.provider_id
ORDER BY s.full_name NULLS LAST, p.full_name
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== Provider hierarchy (provider → supervisor) ===
 provider_id             provider          specialty           supervisor supervisor_specialty
          16       Dr. Unassigned          Neurology Dr. Fatima Al-Rashid      Family Medicine
          15    Dr. Marcus Okafor          Radiology       Dr. James Wong             Oncology
          14       Dr. Priya Nair Emergency Medicine       Dr. James Wong             Oncology
          13   Dr. Ethan Tremblay       Orthopaedics    Dr. Sarah MacNeil           Cardiology
          12 Dr. Fatima Al-Rashid    Family Medicine    Dr. Sarah MacNeil           Cardiology
          11       Dr. James Wong           Oncology    Dr. Sarah MacNeil           Cardiology
          10    Dr. Sarah MacNeil         Cardiology                 None                 None


---
## Finding peers: patients who share a provider

In [3]:
# Self join on encounters: find pairs of patients seen by the same provider
print("=== Patient pairs seen by the same provider ===")
q = """
SELECT  e1.provider_id,
        prov.full_name                        AS provider,
        p1.first_name || ' ' || p1.last_name  AS patient_a,
        p2.first_name || ' ' || p2.last_name  AS patient_b
FROM    encounters AS e1
JOIN    encounters AS e2
    ON  e1.provider_id = e2.provider_id
    AND e1.patient_id  < e2.patient_id   -- avoid duplicates and self-pairs
JOIN    patients  AS p1   ON p1.patient_id  = e1.patient_id
JOIN    patients  AS p2   ON p2.patient_id  = e2.patient_id
JOIN    providers AS prov ON prov.provider_id = e1.provider_id
ORDER BY prov.full_name, patient_a
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("Note: e1.patient_id < e2.patient_id ensures each pair appears once, not twice.")


=== Patient pairs seen by the same provider ===
 provider_id             provider        patient_a        patient_b
          13   Dr. Ethan Tremblay    James MacLeod     Sofia Petrov
          13   Dr. Ethan Tremblay    James MacLeod    Marcus Okafor
          13   Dr. Ethan Tremblay     Sofia Petrov    Marcus Okafor
          12 Dr. Fatima Al-Rashid Fatima Al-Rashid   Ethan Tremblay
          12 Dr. Fatima Al-Rashid Fatima Al-Rashid   Ethan Tremblay
          12 Dr. Fatima Al-Rashid        Liam Chen Fatima Al-Rashid
          12 Dr. Fatima Al-Rashid        Liam Chen Fatima Al-Rashid
          12 Dr. Fatima Al-Rashid        Liam Chen   Ethan Tremblay
          14       Dr. Priya Nair      Aroha Ngata     Sofia Petrov
          10    Dr. Sarah MacNeil      Aroha Ngata        Liam Chen
          10    Dr. Sarah MacNeil      Aroha Ngata    Noah Williams
          10    Dr. Sarah MacNeil      Aroha Ngata        Mei Zhang
          10    Dr. Sarah MacNeil      Aroha Ngata       Priya Nair


---
## Referral self join: who refers to whom

In [4]:
# Referrals table has from_provider and to_provider — both reference providers
print("=== Referral network: from provider → to provider ===")
q = """
SELECT  r.referral_id,
        r.referral_date,
        src.full_name   AS referring_provider,
        src.specialty   AS referring_specialty,
        dst.full_name   AS receiving_provider,
        dst.specialty   AS receiving_specialty,
        p.first_name || ' ' || p.last_name AS patient,
        r.reason
FROM    referrals  AS r
JOIN    providers  AS src ON r.from_provider = src.provider_id
JOIN    providers  AS dst ON r.to_provider   = dst.provider_id
JOIN    patients   AS p   ON r.patient_id    = p.patient_id
ORDER BY r.referral_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Referral summary: who receives the most referrals?
print()
print("=== Referral receipt count by provider ===")
q2 = """
SELECT  dst.full_name   AS provider,
        dst.specialty,
        COUNT(r.referral_id) AS referrals_received
FROM    providers  AS dst
LEFT JOIN referrals AS r ON r.to_provider = dst.provider_id
GROUP BY dst.provider_id, dst.full_name, dst.specialty
ORDER BY referrals_received DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Referral network: from provider → to provider ===
 referral_id referral_date   referring_provider referring_specialty receiving_provider receiving_specialty          patient                              reason
           2    2023-03-01    Dr. Sarah MacNeil          Cardiology     Dr. James Wong            Oncology        Liam Chen                Suspected malignancy
           1    2023-03-10 Dr. Fatima Al-Rashid     Family Medicine  Dr. Sarah MacNeil          Cardiology Fatima Al-Rashid                Chest pain follow-up
           5    2023-06-15    Dr. Sarah MacNeil          Cardiology  Dr. Marcus Okafor           Radiology    Noah Williams              Imaging for chest pain
           3    2023-09-05       Dr. Priya Nair  Emergency Medicine  Dr. Sarah MacNeil          Cardiology       Priya Nair                           AF workup
           4    2023-12-01 Dr. Fatima Al-Rashid     Family Medicine Dr. Ethan Tremblay        Orthopaedics     Sofia Petrov Back pain unresponsive

---
## Finding duplicate rows with a self join

In [5]:
# Find patients with the same date of birth (possible duplicates)
# Real-world EHR dedup: same name + DOB often indicates duplicate patient records
print("=== Patients sharing the same date of birth ===")
q = """
SELECT  a.patient_id  AS id_a,
        a.first_name || ' ' || a.last_name AS name_a,
        b.patient_id  AS id_b,
        b.first_name || ' ' || b.last_name AS name_b,
        a.dob
FROM    patients AS a
JOIN    patients AS b
    ON  a.dob        = b.dob
    AND a.patient_id < b.patient_id   -- each pair once
ORDER BY a.dob
"""
result = pd.read_sql(q, conn)
if len(result) == 0:
    print("No patients share a DOB in this dataset.")
    print("Inserting a test duplicate...")
    conn.execute("INSERT INTO patients VALUES (13,'Aroha','Duplicate','1985-03-12','ON',1)")
    conn.commit()
    print(pd.read_sql(q, conn).to_string(index=False))
else:
    print(result.to_string(index=False))

conn.close()


=== Patients sharing the same date of birth ===
No patients share a DOB in this dataset.
Inserting a test duplicate...
 id_a      name_a  id_b          name_b        dob
    1 Aroha Ngata    13 Aroha Duplicate 1985-03-12


---
## Common Pitfalls

**1. Forgetting `a.id < b.id` produces every pair twice plus self-pairs**
`JOIN t AS b ON a.dept = b.dept` without a row-ordering condition matches A→B and B→A as separate rows, and also A→A (self-pairs). Use `a.id < b.id` to get each unique pair exactly once and exclude self-matches. Use `a.id <> b.id` if you only want to exclude self-matches but don't mind seeing A→B and B→A.

**2. Alias confusion with three-way self-join**
When joining the same table twice in one query (e.g. provider → supervisor → department head), use highly descriptive aliases: `providers AS p`, `providers AS sup`, `providers AS dept_head`. Short aliases like `a`, `b`, `c` become unreadable quickly.

**3. Self-joins on large tables without indexes are very slow**
A self join effectively does an N² comparison. On a 1M-row table without an index on the join key, this is catastrophic. Always ensure the join column is indexed before running a self join on production data.

**4. Hierarchy traversal with a simple self join is limited to one level**
`JOIN providers AS s ON p.supervisor_id = s.provider_id` only traverses one level of hierarchy. To traverse arbitrary depths (the full management chain), use a recursive CTE — covered in `05_subqueries_ctes`.

**5. Self join on a floating-point column causes missed matches**
`JOIN t AS b ON a.measurement = b.measurement` on REAL columns will miss matches due to floating-point precision. For numeric comparisons in self joins, use a tolerance: `ABS(a.value - b.value) < 0.001`, or round first.


---
*sql_methods_library - Samantha McGarrigle*